[![pythonista.io](img/pythonista.png)](https://www.pythonista.io)

# *Terraform* para *Artifact Registry* y Google *Cloud Functions*.

Esta sesión formaliza el aprovisionamiento de destino productivo en *GCP* para que el *pipeline* no dependa de infraestructura creada manualmente.

## Objetivos.

- Modelar recursos base para una ruta de despliegue en *GCP*.
- Relacionar artefactos versionados con el entorno de ejecución en prod.
- Entender el papel de Google *Artifact Registry* y *Cloud Functions* como runtime gestionado.

## Puente rápido de *Terraform* (si partes de cero).

Antes de modelar recursos en *GCP*, valida este mapa mental:

- *Terraform* describe infraestructura deseada, no pasos manuales.
- `plan` te deja revisar cambios antes de ejecutar.
- `apply` solo debe correr con aprobación y contexto de entorno.
- El `state` remoto evita colisiones cuando trabajan varias personas o *pipelines*.

En esta sesión nos enfocamos en traducir componentes de despliegue (registry, service account y function) a recursos declarativos versionados.

## Recursos típicos en el codebase actual.

La base de *Terraform* incluida en este repositorio modela al menos:

- un repositorio de artefactos e imágenes en Google [*Artifact Registry*](https://cloud.google.com/artifact-registry);
- una instancia de [*Cloud SQL*](https://cloud.google.com/sql) PostgreSQL;
- base y usuario de aplicación en *Cloud SQL*;
- variables y secretos separados para test y prod mediante *GitHub* Environments.

El objetivo es que la entrega continua tenga un destino reproducible y no dependiente de configuración manual.

## Relación entre artefacto, base de datos y entorno.

En esta ruta, prod vive en *GCP* con control de identidad federada por *OIDC*. El entorno *Linux* de test se trata en cuadernos posteriores como ruta didáctica de operación sobre host.

*Artifact Registry* permite mantener trazabilidad entre *commit*, *build* y despliegue sin reconstruir imagen por entorno.

**Nota:** el despliegue serverless (*Cloud Functions* 2nd gen / *Cloud Run*) puede consumir estos recursos base y se integra como capa de despliegue sobre la infraestructura parametrizada.

## *Cloud Functions* 2nd gen: ejemplo de recurso *Terraform*.

[*Cloud Functions*](https://cloud.google.com/functions) 2nd gen se despliega sobre [*Cloud Run*](https://cloud.google.com/run). El recurso *HCL* equivalente es:

```hcl
resource "google_cloudfunctions2_function" "app" {
  name     = var.app_name
  location = var.region

  build_config {
    runtime     = "python312"
    entry_point = "app"
    source {
      storage_source {
        bucket = google_storage_bucket.source.name
        object = google_storage_bucket_object.source.name
      }
    }
  }

  service_config {
    min_instance_count    = 0
    max_instance_count    = 3
    available_memory      = "256M"
    service_account_email = google_service_account.cicd_deployer.email

    environment_variables = {
      DB_CONNECTION = google_sql_database_instance.app.connection_name
    }
  }
}
```

Puntos clave:

- `build_config` define origen (bucket) y runtime; el *build* ocurre en *GCP*, no en el *runner*.
- `service_config.service_account_email` aplica mínimo privilegio: la función corre con el SA de despliegue, no con la cuenta del proyecto.
- `min_instance_count = 0` implica cold start; ajustar a 1 para servicios con SLA de latencia exigente.


## Puente con *runners* y despliegue.

La decisión de *runners* se mantiene consistente con el curso:

- hosted como opción por defecto para orquestar *pipeline*;
- self-hosted solo cuando una etapa requiere red privada o controles adicionales.

Esto permite sostener un mismo proceso para test *Linux* y prod *GCP* sin duplicar lógica de *build*.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>